In [1]:
from pathlib import Path
import sys

# Get the project root directory.
PROJECT_ROOT = Path.cwd().resolve().parent

# Add the project root to Python's import path.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

import os

# SSL certificate configuration:
# Uses the Linux system's trusted CA certificates for HTTPS connections.
# This is required in our corporate network because the default Python
# certificate bundle does not trust the network's certificate chain.
os.environ['REQUESTS_CA_BUNDLE'] = '/etc/ssl/certs/ca-certificates.crt'
os.environ['SSL_CERT_FILE'] = '/etc/ssl/certs/ca-certificates.crt'

Project root: /home/nineleaps/Documents/da_python/.venv/nexa-assist


In [3]:
from src.rag.pipeline import RAGPipeline

print('RAGPipeline imported successfully.')

RAGPipeline imported successfully.


In [4]:
rag = RAGPipeline()

print('RAG pipeline loaded successfully.')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2597.65it/s]


RAG pipeline loaded successfully.


### RAG Retrieval Testing

This section tests whether the RAG pipeline retrieves the correct policy information for different employee questions. Each question is converted into an embedding and compared against the policy chunks stored in FAISS. The retrieved source, chunk number, and similarity distance are displayed to verify that relevant information is being returned. This testing is performed before agent integration so that retrieval problems can be isolated from agent-routing problems.


In [5]:
test_questions = [
    'How many WFH days can I take per month?',
    'How many casual leaves can an employee take in a year?',
    'What is the hotel reimbursement limit for domestic travel?',
    'When should travel expenses be submitted?',
    'What expenses are not reimbursable?',
    'Can I apply for leave one day before taking it?',
    'What are the standard working hours while working from home?',
    'How much casual leave do I personally have left?'
]

for question in test_questions:

    print('=' * 80)
    print(f'Question: {question}')
    print()

    results = rag.retrieve(question)

    for result in results:
        print(f"Source: {result['source']}")
        print(f"Chunk: {result['chunk_id']}")
        print(f"Distance: {result['distance']:.4f}")
        print(f"Content: {result['text'][:300]}...")
        print()

Question: How many WFH days can I take per month?

Source: wfh_policy.pdf
Chunk: 0
Distance: 1.0778
Content: NEXACORE TECHNOLOGIES PVT. LTD.
WORK FROM HOME POLICY
Policy ID: NC-HR-002
Effective Date: 2026-01-01
1. ELIGIBILITY
Employees may use up to 8 Work From Home (WFH) days per calendar month.
WFH is intended for focused work, personal logistics, or temporary commuting
constraints.
Certain office-based ...

Source: leave_policy.pdf
Chunk: 1
Distance: 1.2316
Content: For this NexaAssist demo, leave duration is calculated using calendar days.
Employees cannot request more leave days than their available balance.
3. WORK FROM HOME AND LEAVE
WFH is governed by the separate Work From Home Policy.
A WFH day is not deducted from Casual Leave, Earned Leave, or Sick Lea...

Source: wfh_policy.pdf
Chunk: 1
Distance: 1.3506
Content: 3. WORKING HOURS
Standard working hours are 9:30 AM to 6:30 PM, Monday to Friday, unless the
employee's team follows a different schedule.
Employees must attend re

### Targeted Retrieval Validation

This test focuses on questions where the exact policy rule is important for the final answer. The complete retrieved chunk is displayed instead of truncating the text, making it possible to verify whether the required rule is actually present in the retrieved context. This helps distinguish a correct document match from a truly useful retrieval result before the RAG pipeline is connected to the agent.


In [6]:
question = 'Can I apply for leave one day before taking it?'

results = rag.retrieve(question)

for result in results:
    print('=' * 80)
    print(f"Source: {result['source']}")
    print(f"Chunk: {result['chunk_id']}")
    print(f"Distance: {result['distance']:.4f}")
    print()
    print(result['text'])
    print()

Source: leave_policy.pdf
Chunk: 1
Distance: 1.0719

For this NexaAssist demo, leave duration is calculated using calendar days.
Employees cannot request more leave days than their available balance.
3. WORK FROM HOME AND LEAVE
WFH is governed by the separate Work From Home Policy.
A WFH day is not deducted from Casual Leave, Earned Leave, or Sick Leave
unless the employee also submits a leave request.
4. LEAVE CANCELLATION
A submitted leave request may be cancelled before its start date.
Cancelled leave requests do not reduce the employee's leave balance.
5. NEXAASSIST SCOPE
NexaAssist can:
- Answer leave policy questions.
- Check an employee's available leave balance.
- Validate a leave request.
- Submit a leave request after employee confirmation.
NexaAssist does not approve leave on behalf of a manager or HR team.

Source: leave_policy.pdf
Chunk: 0
Distance: 1.0747

NEXACORE TECHNOLOGIES PVT. LTD.
EMPLOYEE LEAVE POLICY
Policy ID: NC-HR-001
Effective Date: 2026-01-01
1. LEAVE CATEGOR

## Step 1: Import Required Libraries

This step imports the libraries required by the RAG pipeline.
The notebook acts as a demonstration layer for the reusable Python modules.
Keeping implementation logic inside `src/rag` allows the agent to reuse the same components later.
The notebook therefore focuses on demonstrating the pipeline and inspecting its results.

In [ ]:
from src.rag.config import (
    DOCUMENTS_PATH,
    VECTORSTORE_PATH,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
    EMBEDDING_MODEL,
    TOP_K
)

from src.rag.document_loader import load_documents
from src.rag.chunker import create_chunks
from src.rag.embeddings import EmbeddingModel
from src.rag.vector_store import VectorStore
from src.rag.retriever import PolicyRetriever

## Step 3: Split Documents into Chunks

Large policy documents are divided into smaller overlapping sections called chunks.
Chunking improves retrieval because only relevant sections need to be searched and provided to the LLM.
`RecursiveCharacterTextSplitter` attempts to preserve natural text boundaries while respecting the chunk size.
Source metadata is retained so retrieved information can later be traced back to its policy document.

In [ ]:
from src.rag.document_loader import load_documents
from src.rag.chunker import create_chunks
from src.rag.config import DOCUMENTS_PATH

documents = load_documents(DOCUMENTS_PATH)

chunks = create_chunks(documents)

print(f'Total documents: {len(documents)}')
print(f'Total chunks: {len(chunks)}')

In [ ]:
for chunk in chunks[:3]:
    print(f"Source: {chunk['source']}")
    print(f"Chunk ID: {chunk['chunk_id']}")
    print(f"Text: {chunk['text'][:300]}...")
    print('-' * 60)

## Step 4: Generate Embeddings

Embeddings convert each text chunk into a numerical representation of its meaning.
The same embedding model is used for both policy chunks and employee questions.
This allows semantic similarity to be calculated between a query and stored policy content.
The selected `all-MiniLM-L6-v2` model generates 384-dimensional vectors.

In [ ]:
from src.rag.embeddings import EmbeddingModel

embedding_model = EmbeddingModel()

texts = [chunk['text'] for chunk in chunks]

embeddings = embedding_model.encode_documents(texts)

print(f'Number of chunks: {len(texts)}')
print(f'Embedding shape: {embeddings.shape}')

## Step 5: Build the FAISS Vector Store

FAISS (Facebook AI Similarity Search) is a library for efficiently searching numerical vectors.
The generated embeddings are stored in a FAISS index so similar policy content can be found quickly.
`IndexFlatL2` compares vectors using L2 distance, where smaller distance means greater similarity.
The original chunks and their metadata are stored separately because the FAISS index contains vectors only.

## Step 5.3: Create the Policy Vector Index

The indexing process combines document loading, chunking, and embedding generation into one workflow.
It runs when the policy knowledge base is created or when policy documents are updated.
The resulting FAISS index can then be reused for multiple employee queries without regenerating embeddings.
This separates the expensive indexing stage from the faster retrieval stage.

### Step 6 — Policy Retrieval

The retriever connects a user question to the FAISS vector store created earlier.
The question is converted into the same 384-dimensional embedding space used for the policy chunks.
FAISS then returns the chunks that are most semantically similar to the question.
Each result retains its source document and chunk ID for traceability and citations.
This step retrieves evidence only; generating the final answer will be handled by the agent later.

In [ ]:
from src.rag.retriever import PolicyRetriever

# Create the policy retriever.
retriever = PolicyRetriever()

# Load the previously created FAISS index.
retriever.load()

# Search the company policies using a sample employee question.
results = retriever.retrieve(
    'How many WFH days can I take per month?'
)

# Display the retrieved policy chunks.
for result in results:
    print(f"Source: {result['source']}")
    print(f"Chunk: {result['chunk_id']}")
    print(f"Distance: {result['distance']:.4f}")
    print(result['text'])
    print('-' * 80)

### Retrieval Testing

The retrieval system is tested using representative employee questions.
The purpose is to verify that semantically similar policy information is retrieved correctly.
Questions cover different company policies such as leave, WFH, travel, and reimbursement.
The retrieved source documents and similarity distances are inspected to evaluate retrieval quality.
Employee-specific questions are also included to verify that the RAG system does not retrieve unrelated policy information.

In [ ]:
# Create the RAG pipeline.
rag = RAGPipeline()

# Questions representing different company policy scenarios.
test_questions = [
    'How many WFH days can I take per month?',
    'How many casual leaves can an employee take in a year?',
    'What is the hotel reimbursement limit for domestic travel?',
    'When should travel expenses be submitted?',
    'What expenses are not reimbursable?',
    'Can I apply for leave one day before taking it?',
    'What are the standard working hours while working from home?',
    'How much casual leave do I personally have left?'
]

# Retrieve and display the top results for every question.
for question in test_questions:

    print(f'\nQuestion: {question}')
    print('=' * 80)

    results = rag.retrieve(question)

    for rank, result in enumerate(results, start=1):
        print(f'\nResult {rank}')
        print(f'Source: {result["source"]}')
        print(f'Chunk ID: {result["chunk_id"]}')
        print(f'Distance: {result["distance"]:.4f}')
        print(f'Text: {result["text"]}')